# Generate Structured Outputs
In this case I am interested in generating JSON. I want the model to find out what topic the user query is about. I would like to use this to do similarity search from a different collection depending on the topic


## Method

In [ ]:
%pip install -qU langchain-ollama --quiet

Import libraries

In [4]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

Initializing the model

In [5]:
llm = ChatOllama(
    model="gemma4:e4b",
    temperature=0.1,
)

Create the prompt template

In [11]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant."),
    ("user", """You are an expert data extraction engine. Your sole output must be a JSON object. Do not include any introduction, explanation, or markdown outside of the JSON block.

Task: Read the following text and extract the topic of the query. Possible topics are: [technology, cooking, other]

Text: {input}

JSON Format:
{{
  "topic": "string"
}}

Output JSON only:""")
    ])

Create the chain using ```LCEL``` syntax

In [54]:
chain = prompt | llm

## Test it
Test the chain with "invoke" method

In [55]:
response = chain.invoke({"input": "what is the colour of the sky?"})
print(response.content)

{
  "topic": "other"
}


In [56]:
response = chain.invoke({"input": "What are the ingredients of bolognese sauce?"})
print(response.content)

{
  "topic": "cooking"
}


In [57]:
response = chain.invoke({"input": "How many layers in the OSI model?"})
print(response.content)

{
  "topic": "technology"
}


The output is a JSON string. We can convert it to a dictionary to do something with it downstrean. We could use the built-in "json" library

In [58]:
import json
topic_dict = json.loads(response.content)

In [59]:
print("Original response type :", type(response.content))
print("Converted response type:", type(topic_dict))
topic_dict

Original response type : <class 'str'>
Converted response type: <class 'dict'>


{'topic': 'technology'}

Or we can use a Langchain output parser, although this requires installing additional packages vs python's json which is there by default

In [60]:
!pip install langchain langchain-community --quiet


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\DELL\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


New imports and new chain that includes the output parser

In [61]:
from langchain_core.output_parsers import JsonOutputParser
chain2 = prompt | llm | JsonOutputParser()

The response is now a dictionary

In [62]:

response = chain2.invoke({"input": "How many layers in the OSI model?"})
print(response)
print(type(response))


{'topic': 'technology'}
<class 'dict'>
